# LSEG Data Pull — NetPayout (Standard Puller)

Dieses Notebook nutzt **nur** den `standard_lseg_series_puller`.

Module:
- `NP1`: Balance Sheet (BE, assets, debt)
- `NP2`: Income Statement (Sales, NetIncome, GrossProfit, Cogs)
- `NP3`: Payout + Liquidity (Dividends, Buybacks, CashSTInvst)


## 0) Setup


In [1]:
from pathlib import Path
import shutil
import warnings

import numpy as np
import pandas as pd

from standard_lseg_series_puller import (
    SeriesPullConfig,
    SeriesFieldSpec,
    run_standard_pull,
)

warnings.filterwarnings("ignore", category=FutureWarning, module="lseg")
pd.set_option("display.max_columns", 200)


## 1) Input + Run-Konfiguration


In [2]:
BASE_DIR = Path('/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data')
DATA_DIR = BASE_DIR / 'intermediate'
CACHE_DATA_DIR = BASE_DIR / 'cache'
CACHE_DATA_DIR.mkdir(parents=True, exist_ok=True)

BASE_PATH = DATA_DIR / 'euro500.parquet'
OUTPUT_PATH = DATA_DIR / 'euro500_netpayout.parquet'

if not BASE_PATH.exists():
    raise FileNotFoundError(f'Missing file: {BASE_PATH}')

base = pd.read_parquet(BASE_PATH).copy()
if 'date' not in base.columns or 'firm_id' not in base.columns:
    raise ValueError('euro500.parquet must contain at least firm_id and date columns.')

print('Loaded base:', BASE_PATH)
print('rows:', len(base), '| companies:', base['firm_id'].nunique())
print('date range:', pd.to_datetime(base['date'], errors='coerce').min(), '->', pd.to_datetime(base['date'], errors='coerce').max())

# ---------- Run control ----------
# Für komplettes Neuziehen: beide Flags auf True lassen.
RESET_PULL_STATE = False
FORCE_REFRESH_ALL = False
RUN_LSEG_PULL = True
BATCH_SIZE = 10
ASOF_TOL_DAYS = 365
DEBUG_RAW_FIRST_N = 0
PRE_INDEX_PREFETCH_YEARS = 2

# Wenn True, wird auch das finale Output-Parquet entfernt und neu aufgebaut.
RESET_OUTPUT_FILE = False





Loaded base: /Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data/intermediate/euro500.parquet
rows: 56500 | companies: 1248
date range: 1997-12-31 00:00:00 -> 2025-12-31 00:00:00


## 2) Reset (optional, für kompletten Neu-Pull)


In [3]:
MODULES = {
    'NP1': {
        'cache_dir': CACHE_DATA_DIR / 'np1_cache_by_company_id',
        'step_rows_path': CACHE_DATA_DIR / 'np1_step_rows.parquet',
        'checkpoint_path': CACHE_DATA_DIR / 'np1_checkpoint.json',
        'bad_ids_path': CACHE_DATA_DIR / 'np1_bad_ids.csv',
        'bad_rows_log_path': CACHE_DATA_DIR / 'np1_bad_rows.parquet',
    },
    'NP2': {
        'cache_dir': CACHE_DATA_DIR / 'np2_cache_by_company_id',
        'step_rows_path': CACHE_DATA_DIR / 'np2_step_rows.parquet',
        'checkpoint_path': CACHE_DATA_DIR / 'np2_checkpoint.json',
        'bad_ids_path': CACHE_DATA_DIR / 'np2_bad_ids.csv',
        'bad_rows_log_path': CACHE_DATA_DIR / 'np2_bad_rows.parquet',
    },
    'NP3': {
        'cache_dir': CACHE_DATA_DIR / 'np3_cache_by_company_id',
        'step_rows_path': CACHE_DATA_DIR / 'np3_step_rows.parquet',
        'checkpoint_path': CACHE_DATA_DIR / 'np3_checkpoint.json',
        'bad_ids_path': CACHE_DATA_DIR / 'np3_bad_ids.csv',
        'bad_rows_log_path': CACHE_DATA_DIR / 'np3_bad_rows.parquet',
    },
}

if RESET_PULL_STATE:
    print('Resetting NP pull state...')
    for name, m in MODULES.items():
        if m['cache_dir'].exists():
            shutil.rmtree(m['cache_dir'])
            print(f'  removed cache dir: {m["cache_dir"]}')
        for k in ['step_rows_path', 'checkpoint_path', 'bad_ids_path', 'bad_rows_log_path']:
            fp = m[k]
            if fp.exists():
                fp.unlink()
                print(f'  removed {name} {k}: {fp}')

if RESET_OUTPUT_FILE and OUTPUT_PATH.exists():
    OUTPUT_PATH.unlink()
    print('Removed old output:', OUTPUT_PATH)


## 3) Standard Puller Setup


In [4]:
def run_np_module(
    source_df: pd.DataFrame,
    module_name: str,
    specs: tuple[SeriesFieldSpec, ...],
    primary_output_col: str,
) -> dict:
    m = MODULES[module_name]

    cfg = SeriesPullConfig(
        batch_size=BATCH_SIZE,
        asof_tolerance_days=ASOF_TOL_DAYS,
        prefetch_start_days=int(PRE_INDEX_PREFETCH_YEARS * 365),
        debug_raw_first_n=DEBUG_RAW_FIRST_N,
        force_refresh=FORCE_REFRESH_ALL,
        cache_only=(not RUN_LSEG_PULL),
        skip_known_bad_ids=True,
        max_retries=3,
        base_sleep_sec=0.7,
        series_specs=specs,
        primary_output_col=primary_output_col,
    )

    print('\n' + '=' * 90)
    print(f'RUN {module_name}')
    print('=' * 90)

    res = run_standard_pull(
        pull_type='series',
        source_df=source_df,
        config=cfg,
        cache_dir=m['cache_dir'],
        step_rows_path=m['step_rows_path'],
        checkpoint_path=m['checkpoint_path'],
        bad_ids_path=m['bad_ids_path'],
        bad_rows_log_path=m['bad_rows_log_path'],
        skip_filled_primary=False,
        merge_back=True,
        diag_prefix=f'{module_name.lower()}_',
    )

    print(f"{module_name} stats:", res['stats'])
    return res


NP1_SPECS = (
    SeriesFieldSpec(output_col='BE', fields=('TR.F.COMEQTOT(Period=FY0)',)),
    SeriesFieldSpec(output_col='assets', fields=('TR.F.TOTASSETS(Period=FY0)',)),
    SeriesFieldSpec(output_col='debt', fields=('TR.F.DEBTTOT(Period=FY0)',)),
)

NP2_SPECS = (
    SeriesFieldSpec(output_col='Sales', fields=('TR.F.TotRevenue(Period=FY0)',)),
    SeriesFieldSpec(output_col='NetIncome', fields=('TR.F.NetIncAfterTax(Period=FY0)',)),
    SeriesFieldSpec(output_col='GrossProfit', fields=('TR.F.GrossProfIndPropTot(Period=FY0)',)),
    SeriesFieldSpec(output_col='Cogs', fields=('TR.F.COGSTot(Period=FY0)',)),
)

# CashSTInvst ist bewusst in NP3 integriert.
NP3_SPECS = (
    SeriesFieldSpec(output_col='Dividends', fields=('TR.F.DivPaidCashTotCF(Period=FY0)',)),
    SeriesFieldSpec(output_col='Buybacks', fields=('TR.F.ComStockBuybackNet(Period=FY0)',)),
    SeriesFieldSpec(output_col='CashSTInvst', fields=('TR.F.CashSTInvst(Period=FY0)',)),
)


def coverage_by_quarter(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    x = df.copy()
    x['date'] = pd.to_datetime(x['date'], errors='coerce').dt.normalize()
    x = x.dropna(subset=['firm_id', 'date']).copy()
    x['quarter'] = x['date'].dt.to_period('Q').dt.to_timestamp(how='end').dt.normalize()

    rows = []
    for q, g in x.groupby('quarter', sort=True):
        rec = {
            'quarter': q,
            'n_obs': int(len(g)),
            'n_firms': int(g['firm_id'].nunique()),
        }
        for col in cols:
            if col in g.columns:
                rec[f'cov_{col}_pct'] = round(float(pd.to_numeric(g[col], errors='coerce').notna().mean() * 100.0), 2)
            else:
                rec[f'cov_{col}_pct'] = np.nan
        rows.append(rec)

    out = pd.DataFrame(rows).sort_values('quarter').reset_index(drop=True)
    return out







## 4) NP1 (Balance Sheet)


In [5]:
np1 = run_np_module(base, 'NP1', NP1_SPECS, primary_output_col='BE')
np1_df = np1['merged_df'].copy()
print('NP1 done: rows=', len(np1_df), '| companies=', np1_df['firm_id'].nunique())



RUN NP1

Standard Series Pull Overview
series_specs: BE<-['TR.F.COMEQTOT(Period=FY0)'], assets<-['TR.F.TOTASSETS(Period=FY0)'], debt<-['TR.F.DEBTTOT(Period=FY0)']
request_rows: 56,500
coverage: all_companies=1,248 | full_coverage=93 | partial_coverage=67 | bad_ids=40 | remaining=1,048
mode: CACHE+NETWORK | batch_size: 10
[BATCH 1/105] companies=10 idx=1-10
[BATCH 1/105] [1/1048] firm_id=FIRM0000218 | cand_used=2/2 | unresolved=4 | found_BE=0 | found_assets=0 | found_debt=0 | range_in_index=1997-12-31:1998-09-30 | pulled_range=NA:NA | tried_ids: ISIN:FR0000053035 | RIC:GEAP.PA
[BATCH 1/105] [2/1048] firm_id=FIRM0000219 | cand_used=3/3 | unresolved=3 | found_BE=0 | found_assets=0 | found_debt=0 | range_in_index=1997-12-31:1998-06-30 | pulled_range=NA:NA | tried_ids: ISIN:FR0000050395 | RIC:CROS.PA | RIC:GEDG.PA
[BATCH 1/105] [3/1048] firm_id=FIRM0000220 | cand_used=4/4 | unresolved=52 | found_BE=25 | found_assets=25 | found_debt=25 | range_in_index=1997-12-31:2019-12-31 | pulled_range=2

### NP1 Coverage nach Quartal


In [6]:
np1_cov_q = coverage_by_quarter(np1_df, ['BE', 'assets', 'debt'])
np1_cov_q

,quarter,n_obs,n_firms,cov_BE_pct,cov_assets_pct,cov_debt_pct
0,1997-12-31,500,500,62.2,62.4,59.8
1,1998-03-31,500,500,66.4,66.6,64.4
2,1998-06-30,500,500,68.8,69.2,67.0
3,1998-09-30,500,500,70.8,71.2,69.2
4,1998-12-31,500,500,88.4,88.6,85.4
...,...,...,...,...,...,...
108,2024-12-31,500,500,97.6,97.6,97.2
109,2025-03-31,500,500,98.8,98.8,98.4
110,2025-06-30,500,500,99.0,99.0,98.6
111,2025-09-30,500,500,98.6,98.6,98.2


## 5) NP2 (Income Statement)


In [ ]:
_np2_input = np1_df if 'np1_df' in globals() else base
np2 = run_np_module(_np2_input, 'NP2', NP2_SPECS, primary_output_col='Sales')
np2_df = np2['merged_df'].copy()
print('NP2 done: rows=', len(np2_df), '| companies=', np2_df['firm_id'].nunique())



RUN NP2

Standard Series Pull Overview
series_specs: Sales<-['TR.F.TotRevenue(Period=FY0)'], NetIncome<-['TR.F.NetIncAfterTax(Period=FY0)'], GrossProfit<-['TR.F.GrossProfIndPropTot(Period=FY0)'], Cogs<-['TR.F.COGSTot(Period=FY0)']
request_rows: 56,500
coverage: all_companies=1,248 | full_coverage=0 | partial_coverage=0 | bad_ids=0 | remaining=1,248
mode: CACHE+NETWORK | batch_size: 10
[BATCH 1/125] companies=10 idx=1-10
[BATCH 1/125] [1/1248] firm_id=FIRM0000001 | cand_used=3/3 | unresolved=5 | found_Sales=20 | found_NetIncome=20 | found_GrossProfit=20 | found_Cogs=0 | range_in_index=1997-12-31:2003-12-31 | pulled_range=1998-12-31:2003-09-30 | tried_ids: ISIN:DE0005009740 | ISIN:DE0005009708 | RIC:AAHG.F
[BATCH 1/125] [2/1248] firm_id=FIRM0000002 | cand_used=3/3 | unresolved=0 | found_Sales=113 | found_NetIncome=113 | found_GrossProfit=113 | found_Cogs=0 | range_in_index=1997-12-31:2025-12-31 | pulled_range=1997-12-31:2025-12-31 | tried_ids: ISIN:NL0000331346 | ISIN:NL0000852564 | RIC

### NP2 Coverage nach Quartal


In [ ]:
np2_cov_q = coverage_by_quarter(np2_df, ['Sales', 'NetIncome', 'GrossProfit', 'Cogs'])
np2_cov_q


## 6) NP3 (Payout + Liquidity)


In [ ]:
_np3_input = np2_df if 'np2_df' in globals() else (np1_df if 'np1_df' in globals() else base)
np3 = run_np_module(_np3_input, 'NP3', NP3_SPECS, primary_output_col='Dividends')
out = np3['merged_df'].copy()
out['date'] = pd.to_datetime(out['date'], errors='coerce').dt.normalize()
out = out.dropna(subset=['firm_id', 'date']).sort_values(['firm_id', 'date']).drop_duplicates(['firm_id', 'date'], keep='last')
out.to_parquet(OUTPUT_PATH, index=False)
euro500_netpayout_df = out.copy()
print('\nSaved output:', OUTPUT_PATH)
print('rows:', len(out), '| companies:', out['firm_id'].nunique())


### NP3 Coverage nach Quartal


In [ ]:
np3_cov_q = coverage_by_quarter(euro500_netpayout_df, ['Dividends', 'Buybacks', 'CashSTInvst'])
np3_cov_q


## 7) Quick Check


In [ ]:
check_cols = [
    'BE', 'assets', 'debt',
    'Sales', 'NetIncome', 'GrossProfit', 'Cogs',
    'Dividends', 'Buybacks', 'CashSTInvst',
]

summary = []
for c in check_cols:
    if c in euro500_netpayout_df.columns:
        nn = int(pd.to_numeric(euro500_netpayout_df[c], errors='coerce').notna().sum())
        summary.append((c, nn))

summary_df = pd.DataFrame(summary, columns=['column', 'non_null'])
summary_df
